# Meet the AI Agents

Run all 5 specialized agents directly from this notebook. Each agent has tools that connect to real AWS services and your RDS SQL Server instance.

| Agent | Tools | What It Monitors |
|---|---|---|
| 📊 Database Health | 14 | CPU, memory, connections, wait events |
| ⚡ Query Performance | 13 | Slow queries, blocking, missing indexes |
| 🔒 Security Audit | 8 | Failed logins, encryption, config changes |
| 💾 Data Lifecycle | 25 | Storage, TempDB, backups, table sizes |
| 🎯 Supervisor | 10 | Orchestrates all agents |

💡 **Edit any prompt** and re-run the cell. Add new cells for your own questions. The agent remembers context within the same section.

---
## Setup

Import common dependencies and configure the model:

In [ ]:
import sys, os
sys.path.insert(0, '/workshop/labs/interactive_agents')

from strands import Agent, tool
from strands.models import BedrockModel
import boto3, pymssql, json

REGION = os.getenv('AWS_REGION', 'us-west-2')
DB_INSTANCE_ID = os.getenv('DB_INSTANCE_ID', 'dbops-infra-sqlserver')
DB_SECRET_ID = os.getenv('DB_SECRET_ID', 'dbops-infra-sqlserver-secret')

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=REGION,
)

def _get_db_connection(database='master'):
    """Helper to get SQL Server connection from Secrets Manager."""
    client = boto3.client('secretsmanager', region_name=REGION)
    secret = client.get_secret_value(SecretId=DB_SECRET_ID)
    creds = json.loads(secret['SecretString'])
    return pymssql.connect(
        server=creds['host'], user=creds['username'],
        password=creds['password'], port=int(creds['port']),
        database=database
    )

print(f"✅ Ready | Region: {REGION} | DB: {DB_INSTANCE_ID}")

---
## 📊 Database Health Agent

Monitors CPU, memory, connections, wait events, IOPS, and latency using CloudWatch and Database Insights.

In [ ]:
@tool
def get_cpu_utilization(minutes: int = 60):
    """Get CPU utilization for the RDS SQL Server instance over the specified time period."""
    cw = boto3.client('cloudwatch', region_name=REGION)
    from datetime import datetime, timedelta
    response = cw.get_metric_statistics(
        Namespace='AWS/RDS', MetricName='CPUUtilization',
        Dimensions=[{'Name': 'DBInstanceIdentifier', 'Value': DB_INSTANCE_ID}],
        StartTime=datetime.utcnow() - timedelta(minutes=minutes),
        EndTime=datetime.utcnow(),
        Period=300, Statistics=['Average', 'Maximum']
    )
    points = sorted(response['Datapoints'], key=lambda x: x['Timestamp'])
    return [{'time': str(p['Timestamp']), 'avg': round(p['Average'], 1), 'max': round(p['Maximum'], 1)} for p in points[-5:]]

@tool
def get_database_connections():
    """Get current number of database connections to the RDS instance."""
    cw = boto3.client('cloudwatch', region_name=REGION)
    from datetime import datetime, timedelta
    response = cw.get_metric_statistics(
        Namespace='AWS/RDS', MetricName='DatabaseConnections',
        Dimensions=[{'Name': 'DBInstanceIdentifier', 'Value': DB_INSTANCE_ID}],
        StartTime=datetime.utcnow() - timedelta(minutes=15),
        EndTime=datetime.utcnow(),
        Period=60, Statistics=['Average']
    )
    points = sorted(response['Datapoints'], key=lambda x: x['Timestamp'])
    if points:
        return f"Current connections: {int(points[-1]['Average'])}"
    return "No data available"

@tool
def get_freeable_memory():
    """Get freeable memory for the RDS instance in MB."""
    cw = boto3.client('cloudwatch', region_name=REGION)
    from datetime import datetime, timedelta
    response = cw.get_metric_statistics(
        Namespace='AWS/RDS', MetricName='FreeableMemory',
        Dimensions=[{'Name': 'DBInstanceIdentifier', 'Value': DB_INSTANCE_ID}],
        StartTime=datetime.utcnow() - timedelta(minutes=15),
        EndTime=datetime.utcnow(),
        Period=60, Statistics=['Average']
    )
    points = sorted(response['Datapoints'], key=lambda x: x['Timestamp'])
    if points:
        mb = round(points[-1]['Average'] / (1024*1024), 0)
        return f"Freeable memory: {mb} MB"
    return "No data available"

health_agent = Agent(
    model=model,
    system_prompt="""You are a Database Health Agent monitoring an RDS SQL Server instance.
Use your tools to check CPU, memory, and connections. Provide concise analysis with recommendations.""",
    tools=[get_cpu_utilization, get_database_connections, get_freeable_memory]
)

print("✅ Database Health Agent ready")

In [ ]:
# 📊 Ask the Health Agent — edit this prompt!
response = health_agent("Why is CPU high right now? Give me current metrics.")
print(response)

In [ ]:
# Try another question (agent remembers context from above)
response = health_agent("How about memory — are we running low?")
print(response)

---
## ⚡ Query Performance Agent

Analyzes slow queries, blocking sessions, and missing indexes using SQL Server DMVs.

In [ ]:
@tool
def get_slow_queries(top_n: int = 5):
    """Get the top N slowest queries from the SQL Server plan cache by average elapsed time."""
    conn = _get_db_connection('DBOpsLab')
    cursor = conn.cursor()
    cursor.execute(f"""
        SELECT TOP {top_n}
            SUBSTRING(qt.text, 1, 200) AS query_text,
            qs.execution_count,
            CAST(qs.total_elapsed_time / qs.execution_count / 1000.0 AS DECIMAL(10,1)) AS avg_duration_ms,
            CAST(qs.total_worker_time / qs.execution_count / 1000.0 AS DECIMAL(10,1)) AS avg_cpu_ms
        FROM sys.dm_exec_query_stats qs
        CROSS APPLY sys.dm_exec_sql_text(qs.sql_handle) qt
        WHERE qs.execution_count > 1
        ORDER BY qs.total_elapsed_time / qs.execution_count DESC
    """)
    results = [{'query': row[0], 'executions': row[1], 'avg_ms': float(row[2]), 'avg_cpu_ms': float(row[3])} for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return results

@tool
def get_blocking_sessions():
    """Check for any currently blocking sessions in SQL Server."""
    conn = _get_db_connection('DBOpsLab')
    cursor = conn.cursor()
    cursor.execute("""
        SELECT blocking_session_id, session_id, wait_type, wait_time,
               SUBSTRING(st.text, 1, 100) as query_text
        FROM sys.dm_exec_requests r
        CROSS APPLY sys.dm_exec_sql_text(r.sql_handle) st
        WHERE blocking_session_id > 0
    """)
    results = [{'blocker': row[0], 'blocked': row[1], 'wait_type': row[2], 'wait_ms': row[3], 'query': row[4]} for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return results if results else "No blocking sessions detected"

@tool
def suggest_indexes():
    """Get missing index suggestions from SQL Server's DMV."""
    conn = _get_db_connection('DBOpsLab')
    cursor = conn.cursor()
    cursor.execute("""
        SELECT TOP 5
            d.statement AS table_name,
            d.equality_columns,
            d.inequality_columns,
            d.included_columns,
            s.user_seeks + s.user_scans AS potential_uses
        FROM sys.dm_db_missing_index_details d
        JOIN sys.dm_db_missing_index_groups g ON d.index_handle = g.index_handle
        JOIN sys.dm_db_missing_index_group_stats s ON g.index_group_handle = s.group_handle
        ORDER BY s.user_seeks + s.user_scans DESC
    """)
    results = [{'table': row[0], 'equality_cols': row[1], 'inequality_cols': row[2], 'include_cols': row[3], 'potential_uses': row[4]} for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return results if results else "No missing index suggestions"

query_agent = Agent(
    model=model,
    system_prompt="""You are a Query Performance Agent for SQL Server.
Analyze slow queries, blocking, and suggest index improvements. Be specific with recommendations.""",
    tools=[get_slow_queries, get_blocking_sessions, suggest_indexes]
)

print("✅ Query Performance Agent ready")

In [ ]:
# ⚡ Ask the Query Agent — edit this prompt!
response = query_agent("What are the top 5 slowest queries and do you have index suggestions?")
print(response)

In [ ]:
# Check for blocking
response = query_agent("Are there any blocking sessions right now?")
print(response)

---
## 🔒 Security Audit Agent

Checks encryption, failed logins, and configuration changes.

In [ ]:
@tool
def check_tde_status():
    """Check Transparent Data Encryption (TDE) status for all databases."""
    conn = _get_db_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT db.name, 
               CASE WHEN dek.encryption_state IS NULL THEN 'Not Encrypted'
                    WHEN dek.encryption_state = 3 THEN 'Encrypted'
                    ELSE 'In Progress' END as status
        FROM sys.databases db
        LEFT JOIN sys.dm_database_encryption_keys dek ON db.database_id = dek.database_id
        WHERE db.database_id > 4
    """)
    results = [{'database': row[0], 'status': row[1]} for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return results

@tool
def check_rds_security_settings():
    """Check RDS security configuration — encryption at rest, SSL, public access."""
    rds = boto3.client('rds', region_name=REGION)
    response = rds.describe_db_instances(DBInstanceIdentifier=DB_INSTANCE_ID)
    db = response['DBInstances'][0]
    return {
        'storage_encrypted': db.get('StorageEncrypted', False),
        'publicly_accessible': db.get('PubliclyAccessible', False),
        'deletion_protection': db.get('DeletionProtection', False),
        'multi_az': db.get('MultiAZ', False),
        'engine_version': db.get('EngineVersion'),
        'auto_minor_upgrade': db.get('AutoMinorVersionUpgrade', False)
    }

security_agent = Agent(
    model=model,
    system_prompt="""You are a Security Audit Agent for RDS SQL Server.
Check encryption, access controls, and compliance settings. Flag any security concerns.""",
    tools=[check_tde_status, check_rds_security_settings]
)

print("✅ Security Audit Agent ready")

In [ ]:
# 🔒 Ask the Security Agent
response = security_agent("Run a security audit — check encryption and access settings")
print(response)

---
## 🎯 Supervisor Agent

Orchestrates all agents above. Routes your question to the right sub-agent(s).

Here we use the **agent-as-a-tool** pattern — each sub-agent becomes a tool the Supervisor can call.

In [ ]:
@tool
def invoke_health_check(question: str):
    """Route health-related questions to the Database Health Agent. Use for CPU, memory, connections, IOPS."""
    response = health_agent(question)
    return str(response)

@tool
def invoke_query_analysis(question: str):
    """Route query performance questions to the Query Performance Agent. Use for slow queries, blocking, indexes."""
    response = query_agent(question)
    return str(response)

@tool
def invoke_security_audit(question: str):
    """Route security questions to the Security Audit Agent. Use for encryption, access, compliance."""
    response = security_agent(question)
    return str(response)

supervisor = Agent(
    model=model,
    system_prompt="""You are the Supervisor Agent coordinating a team of specialized database agents.

Route questions to the appropriate sub-agent:
- Health questions (CPU, memory, connections) → invoke_health_check
- Query performance (slow queries, blocking, indexes) → invoke_query_analysis  
- Security (encryption, access, compliance) → invoke_security_audit

For broad questions like 'health report', call multiple agents and correlate findings.""",
    tools=[invoke_health_check, invoke_query_analysis, invoke_security_audit]
)

print("✅ Supervisor Agent ready (orchestrates Health, Query, Security agents)")

In [ ]:
# 🎯 Ask the Supervisor — it decides which agent(s) to call
response = supervisor("Give me a quick database health summary — CPU, connections, and any security concerns")
print(response)

In [ ]:
# Try a focused question — Supervisor routes to the right agent
response = supervisor("Are there any slow queries causing the high CPU?")
print(response)

---
## Your Turn!

Add cells below and ask any question. The Supervisor will route it to the right agent.

**Ideas:**
- `"Why is the database slow?"`
- `"Check if we have any missing indexes and also verify encryption"`
- `"Give me a full report on database health, performance, and security"`

In [ ]:
# Your question here:
response = supervisor("Why is the database slow?")
print(response)